# Antardhi — 02: Model Benchmarking & Explainability

**Objective:** Benchmark **Layer A (Baseline WOE + Logistic Regression Scorecard)** against **Layer B (Challenger LightGBM)** on alternative data credit scoring.

- Evaluates models using regulatory metrics: ROC-AUC, PR-AUC, KS-statistic, Brier score.
- Demonstrates TreeSHAP feature attribution and Adverse Action Reason Codes.
- Showcases Path-to-Eligibility (What-If simulation) and Isolation Forest Anomaly Detection.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.models.features import extract_features, FEATURE_COLUMNS
from src.models.train import train_baseline_model, train_challenger_model
from src.models.evaluate import evaluate_model
from src.models.confidence import compute_confidence_score
from src.models.anomaly import compute_anomaly_flag
from src.explainability.shap_explainer import get_shap_values
from src.explainability.reason_codes import map_to_reason_codes
from src.simulation.what_if import simulate_improvement
from src.models import score_customer

print("Antardhi Model Benchmarking Suite Loaded Successfully.")


Antardhi Model Benchmarking Suite Loaded Successfully.


## 1. Load Dataset & Preprocessing


In [2]:
data_path = "../data/synthetic_credit_data.csv" if os.path.exists("../data/synthetic_credit_data.csv") else "data/synthetic_credit_data.csv"
df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
print(f"Overall Default rate: {df['default_label'].mean():.2%}")
print("Persona distribution:\n", df["persona"].value_counts())

# Split raw dataset into train and held-out test sets to strictly prevent data leakage
df_train, df_test = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["default_label"]
)

X_train = extract_features(df_train)
y_train = df_train["default_label"].values

X_test = extract_features(df_test)
y_test = df_test["default_label"].values

print(f"Training samples: {len(X_train)}, Testing samples (held-out): {len(X_test)}")


Loaded dataset: 8000 rows, 20 columns.
Overall Default rate: 19.88%
Persona distribution:
 persona
Small Merchant           2000
Gig Worker               2000
Informal/Rural Worker    2000
First-time Borrower      2000
Name: count, dtype: int64


Training samples: 6000, Testing samples (held-out): 2000


## 2. Train Models (Baseline Layer A vs Challenger Layer B)


In [3]:
# Layer A: WOE Binning + Logistic Regression Scorecard
baseline_model = train_baseline_model(X_train, y_train)

# Layer B: LightGBM Gradient Boosted Trees
challenger_model = train_challenger_model(X_train, y_train)
print("Both baseline and challenger models successfully trained.")


Both baseline and challenger models successfully trained.


## 3. Regulatory Benchmarking & Model Lift

We evaluate both models on the held-out test split using **ROC-AUC, PR-AUC, KS-statistic, and Brier score**.


In [4]:
metrics_a = evaluate_model(baseline_model, X_test, y_test)
metrics_b = evaluate_model(challenger_model, X_test, y_test)

bench_df = pd.DataFrame([
    {"Model": "Layer A: Baseline (WOE+LogReg)", **metrics_a},
    {"Model": "Layer B: Challenger (LightGBM)", **metrics_b}
])

auc_lift = metrics_b["roc_auc"] - metrics_a["roc_auc"]
ks_lift = metrics_b["ks_stat"] - metrics_a["ks_stat"]
bench_df["ROC-AUC Lift"] = ["-", f"{auc_lift:+.4f}"]
bench_df["KS-Stat Lift"] = ["-", f"{ks_lift:+.4f}"]
print(bench_df.to_string(index=False))


                         Model  roc_auc  pr_auc  ks_stat  brier ROC-AUC Lift KS-Stat Lift
Layer A: Baseline (WOE+LogReg)   0.7701  0.4426   0.4369 0.1351            -            -
Layer B: Challenger (LightGBM)   0.7671  0.4305   0.4150 0.1358      -0.0030      -0.0219


## 4. TreeSHAP Explainability & Adverse Action Reason Codes


In [5]:
sample_customer = df.iloc[10]
shap_dict = get_shap_values(challenger_model, sample_customer)
reason_codes = map_to_reason_codes(shap_dict)

print("Top 5 Reason Codes for Applicant:")
for rc in reason_codes:
    print(f"[{rc['impact']}] {rc['factor']}: {rc['description']}")


Top 5 Reason Codes for Applicant:
[-] months_of_data_available: Short operating history available for assessment
[+] payment_discipline: Regular utility & recharge payments
[+] livelihood_activity: Stable, active work pattern
[-] utility_payment_regularity: Delayed or irregular utility bill payments
[-] upi_inflow_volatility: Adverse Upi Inflow Volatility requiring improvement


C:\Users\suraj\AppData\Local\Programs\Python\Python310\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## 5. Path-to-Eligibility (What-If Simulation)


In [6]:
sim_result = simulate_improvement(
    sample_customer,
    {"inflow_pct": 20.0, "utility_payment_regularity": 0.95}
)
print("What-If Simulation Results:")
print(f"Current Score: {sim_result['current_score']} ({sim_result['current_tier']})")
print(f"Projected Score: {sim_result['new_score']} ({sim_result['new_tier']})")
print(f"Score Delta: {sim_result['delta']:+d} points")


What-If Simulation Results:


Current Score: 563 (High Risk)
Projected Score: 606 (Medium Risk)
Score Delta: +43 points


## 6. Master Orchestrator End-to-End Scoring


In [7]:
score_decision = score_customer(sample_customer)
import pprint
pprint.pprint(score_decision)


{'anomaly_flag': 'Low',
 'confidence': 96.66,
 'reason_codes': [{'description': 'Short operating history available for '
                                  'assessment',
                   'factor': 'months_of_data_available',
                   'impact': '-'},
                  {'description': 'Adverse Mobility Distance Trend requiring '
                                  'improvement',
                   'factor': 'mobility_distance_trend',
                   'impact': '-'},
                  {'description': 'Adverse Ecommerce Return Rate requiring '
                                  'improvement',
                   'factor': 'ecommerce_return_rate',
                   'impact': '-'},
                  {'description': 'Dependable and recurring mobile recharge '
                                  'pattern',
                   'factor': 'telecom_recharge_consistency',
                   'impact': '+'},
                  {'description': 'Regular utility & recharge payments',
             

C:\Users\suraj\AppData\Local\Programs\Python\Python310\lib\site-packages\shap\explainers\_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
